# RAG 入库与检索基础流程

这份 notebook 按照“文本 -> embedding -> Document -> Chroma collection -> 写入/管理 -> 检索 -> retriever”的逻辑整理。

当前阶段聚焦 LangChain + Chroma，不展开其他向量数据库。

## 1. Embedding 基础

Embedding 的作用是把文本转换成向量。向量数据库存储的核心就是这些向量，以及对应的原始文本和 metadata。

在 RAG 中有两类常见 embedding 操作：

- `embed_query()`：把用户问题转换成查询向量。
- `embed_documents()`：把一批文档文本转换成文档向量。

注意：入库和查询必须使用同一个 embedding 模型，否则向量空间不一致，检索结果会失真。

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
    # normalize_embeddings=True 会做 L2 归一化。
    # 对很多检索场景来说，归一化后余弦相似度和向量点积更容易对应。
)

e:\miniconda\envs\langchain2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5034.47it/s]


In [3]:
query_vector = emb.embed_query("什么是RAG")

print(type(query_vector), len(query_vector))  # len（vector） 来源于embedding模型 不同emb模型的输出维度不同
print(query_vector)

<class 'list'> 384
[-0.038526467978954315, 0.058618251234292984, 0.07145114988088608, -0.010258873924612999, -0.02102752774953842, 0.026349807158112526, 0.09934291988611221, 0.011616132222115993, 0.024152647703886032, -0.03808009251952171, 0.11358541995286942, -0.12874919176101685, 0.09167089313268661, -0.04364248737692833, -0.030671555548906326, 0.05739958956837654, 0.039728403091430664, 0.12299805134534836, -0.01743190735578537, 0.011055991984903812, -0.05870894715189934, 0.033248841762542725, 0.013433157466351986, 0.027593813836574554, 0.00561495078727603, -0.020448975265026093, -0.013871296308934689, 0.018947117030620575, 0.042012639343738556, -0.006030370015650988, -0.0261729396879673, 0.03680314123630524, -0.05874110385775566, -0.043657686561346054, 0.03098626807332039, 0.024305257946252823, -0.029921822249889374, 0.011227753013372421, 0.03685459867119789, 0.036298967897892, -0.027438059449195862, -0.02043355256319046, 0.02172025851905346, -0.08578261733055115, -0.000145051264553

In [4]:
from langchain_core.documents import Document
docs_text = [
    "什么是RAG",
    "RAG 是一种结合了检索和生成的自然语言处理技术。",
]
# embed_documents 接收的是 list[str]，不是 list[Document]。
doc_vectors = emb.embed_documents(docs_text)
print(type(doc_vectors), len(doc_vectors), len(doc_vectors[0]))
print(doc_vectors[0])

<class 'list'> 2 384
[-0.038526471704244614, 0.058618269860744476, 0.07145114988088608, -0.010258871130645275, -0.02102750353515148, 0.02634979411959648, 0.0993429645895958, 0.011616145260632038, 0.024152684956789017, -0.0380801185965538, 0.1135854497551918, -0.12874919176101685, 0.09167086333036423, -0.043642450124025345, -0.03067154996097088, 0.05739960819482803, 0.03972841054201126, 0.12299805134534836, -0.01743190921843052, 0.011055970564484596, -0.05870893597602844, 0.03324883431196213, 0.013433157466351986, 0.0275938231498003, 0.00561498012393713, -0.0204489603638649, -0.013871286064386368, 0.018947092816233635, 0.04201267287135124, -0.006030365824699402, -0.026172904297709465, 0.036803122609853745, -0.05874106287956238, -0.04365772753953934, 0.03098629042506218, 0.02430526539683342, -0.029921824112534523, 0.01122775673866272, 0.03685460984706879, 0.03629891574382782, -0.027438044548034668, -0.020433567464351654, 0.021720269694924355, -0.08578266203403473, -0.0001450344716431573,

## 2. Document 与 metadata

LangChain 的 `Document` 是 RAG 中最常用的文档结构。

- `page_content`：真正参与 embedding 和召回的文本内容。
- `metadata`：文档的附加信息，例如来源、类别、难度、章节、文件名等。

metadata 不一定直接参与语义向量计算，但它对后续过滤检索非常关键。

In [5]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="什么是RAG",
        metadata={"source": "example.txt", "topic": "rag"},
    ),
    Document(
        page_content="RAG 是一种结合了检索和生成的自然语言处理技术。",
        metadata={"source": "example.txt", "topic": "rag"},
    ),
]

print(docs[0].page_content)
print(docs[0].metadata)
""" 
embedding的时候 必须embedding str
入库的时候，可以通过 db.from_documents(docs) 直接传入 Document 对象列表，
底层会自动提取 page_content 进行 embedding metadata 会原封不动地存储在向量数据库中，
便于后续检索时使用
"""

什么是RAG
{'source': 'example.txt', 'topic': 'rag'}


' \nembedding的时候 必须embedding str\n入库的时候，可以通过 db.from_documents(docs) 直接传入 Document 对象列表，\n底层会自动提取 page_content 进行 embedding metadata 会原封不动地存储在向量数据库中，\n便于后续检索时使用\n'

## 3. Chroma collection 创建

在 LangChain 中，`Chroma` 对象通常可以理解为“某个 collection 的操作入口”。
创建的文件中，chroma.sqlite3是SQLite数据库文件，保存向量collection的结构等配置信息

常见创建方式有三种：

- `Chroma(...)`：连接或创建 collection，不会自动写入数据。
- `Chroma.from_texts(...)`：创建 collection，并立即写入纯文本。
- `Chroma.from_documents(...)`：创建 collection，并立即写入 `Document`。

`persist_directory` 是本地持久化目录，`collection_name` 是 collection 名称。

In [6]:
from langchain_chroma import Chroma

# 方式一：连接或创建 collection，不立刻导入数据。
db = Chroma(
    persist_directory="./chroma_db",
    collection_name="my_collection",
    embedding_function=emb,
)

print(db._collection_name)
print(db._collection.count())

my_collection
2


In [7]:
# 方式二：从纯文本创建 collection，并立即写入。
texts = ["文本1", "文本2"]

db_from_texts = Chroma.from_texts(
    texts,
    embedding=emb,
    persist_directory="./chroma_db_from_texts",
    collection_name="my_collection",
)

print(db_from_texts._collection_name)
print(db_from_texts._collection.count())

my_collection
6


In [8]:
# 方式三：从 Document 创建 collection，并立即写入。
docs_for_create = [
    Document(page_content="文本1", metadata={"source": "file1.txt"}),
    Document(page_content="文本2", metadata={"source": "file2.txt"}),
]

db_from_documents = Chroma.from_documents(
    docs_for_create,
    embedding=emb,
    persist_directory="./chroma_db_from_documents",
    collection_name="my_collection",
)

print(db_from_documents._collection_name)
print(db_from_documents._collection.count())

my_collection
6


### 3.1 查看已有 collection

如果要查看某个持久化目录下有哪些 collection，需要使用 Chroma 原生 client。

注意这里应该使用 `chromadb.PersistentClient(path=...)`，而不是普通的 `chromadb.Client()`。

还可以用这种方法清空整个collection

In [9]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collections = client.list_collections()
print([c.name for c in collections])

['my_collection']


In [ ]:
client.delete_collection("my_collection")

## 4. 数据写入与管理

真实项目里，推荐主流程使用 LangChain 封装的 `db.add_documents()`。

底层的 `db._collection.add()` 可以用于理解 Chroma 原生 API，但项目代码里应尽量少直接依赖 `_collection` 这种内部属性。

In [10]:
docs = [
    Document(page_content="什么是RAG", metadata={"source": "example.txt", "topic": "rag"}),
    Document(page_content="RAG 是一种结合了检索和生成的自然语言处理技术。", metadata={"source": "example.txt", "topic": "rag"}),
]

# 使用稳定 ID 可以避免每次运行都生成随机 ID。随机ID可以通过打印之后查看
# 学习阶段反复执行同一个单元时，重复 ID 可能报错，这是正常的；真实项目里要做去重或更新逻辑。
ids = [f"example_{i}" for i in range(len(docs))]

db = Chroma(
    persist_directory="./chroma_db",
    collection_name="my_collection",
    embedding_function=emb,
)

db.add_documents(docs, ids=ids)
print(db._collection.count())

2


# **去重 ID**

### 4.1 底层 add 的写法

下面是 Chroma 底层 API 的写法。学习时可以理解，但不要和上面的 `db.add_documents()` 同时执行同一批数据，否则会重复入库。

In [ ]:
documents = [doc.page_content for doc in docs]
metadatas = [doc.metadata for doc in docs]
ids = [f"example_{i}" for i in range(len(docs))]

# 不要和 db.add_documents(docs, ids=ids) 同时执行同一批数据。
db._collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

### 4.2 更新与删除

更新和删除都依赖 ID。真实项目中如果没有稳定 ID，就很难可靠地更新已有数据。

In [12]:
""" 
db.add_documents    是新增,如果id存在则不会更新(根据自动去重)
db.update_documents 是更新,如果id不存在则会不做操作,不会新增
"""

updated_doc = Document(
    page_content="RAG 是 Retrieval-Augmented Generation，用检索结果增强大模型回答。",
    metadata={"source": "example.txt", "topic": "rag"},
)

# 更新指定 ID 的文档。
db.update_documents(ids=["example_100"], documents=[updated_doc])

# 删除指定 ID 的文档。
db.delete(ids=["example_0"])

#### 查看collection的信息

In [30]:
print(db._collection_name)
print(db._collection.count())

my_collection
5


### 4.3 清空 collection

学习阶段经常需要重置 collection。最直接的方法是删除 collection，再重新创建。

删除是破坏性操作，所以这里默认注释掉。

In [26]:
client = chromadb.PersistentClient(path="./chroma_db")

# 删除整个 collection。执行后 my_collection 里的数据会被清空。
client.delete_collection("my_collection")

# 删除后可以重新连接或创建。
# db = Chroma(
#     persist_directory="./chroma_db",
#     collection_name="my_collection",
#     embedding_function=emb,
# )


client是Chroma底层的官方原生客户端，操作类似于数据库本身
优点是控制力强，可以做一些db不能做的事情

比如清空collection 查看有哪些collection、有多少个

db = Chroma 是langchain封装的向量数据库对象


In [20]:
print(db._client.list_collections())

[Collection(name=my_collection)]


### 4.4 查看 collection 中的数据

可以用 `get()` 查看数据。常见查看方式包括：

- 按 ID 查。
- 分页查。
- 指定 include 返回 documents、metadatas、embeddings。

In [31]:
# 按 ID 查询。
result = db._collection.get(
    # ids=["example_1"],
    include=["documents", "metadatas"] # include代表 我希望出现这些内容 但是似乎这些都是包含的 愚蠢
)
for doc_id , doc_text , mete in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"ID: {doc_id}\n文本: {doc_text}\n元数据: {mete}\n")  

ID: example_1
文本: RAG 是一种结合了检索和生成的自然语言处理技术。
元数据: {'source': 'example.txt', 'topic': 'rag'}

ID: example_2
文本: 什么是RAG
元数据: {'source': 'example.txt', 'topic': 'rag'}

ID: example_3
文本: RAG 是一种结合了检索和生成的自然语言处理技术。
元数据: {'source': 'example.txt', 'topic': 'rag'}

ID: example_5
文本: 什么是RAG
元数据: {'source': 'example.txt', 'topic': 'rag'}

ID: example_6
文本: RAG 是一种结合了检索和生成的自然语言处理技术。
元数据: {'topic': 'rag', 'source': 'example.txt'}



In [22]:
# 分页查看前 5 条。
# 第 1 页
all_data = db._collection.get(
    limit=5,
    offset=0,
    include=["documents", "metadatas"]
)

# 第 2 页
page2 = db._collection.get(
    limit=5,
    offset=5,
    include=["documents", "metadatas"]
)

for doc_id, doc_text, meta in zip(all_data["ids"], all_data["documents"], all_data["metadatas"]):
    print(f"{doc_id}: {doc_text[:30]}..., metadata={meta}")

example_1: RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'topic': 'rag', 'source': 'example.txt'}


## 5. 基础检索

检索阶段就是：把 query 转成向量，然后在 collection 中找最相近的文档。

这里先看 Chroma 底层查询，再看 LangChain 封装查询。

### 5.1 Chroma 底层查询

`db._collection.query()` 是 Chroma 原生查询接口。它可以直接传 `query_texts`，也可以传提前算好的 `query_embeddings`。

学习时可以看它的返回结构；实际 LangChain 项目中，主流程通常优先用 `similarity_search()` 或 retriever。

In [33]:
query_text = "RAG是什么"

results = db._collection.query(
    query_texts=[query_text],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(results) # 可以看到输出的是一个字典 所以要用 result[""]这种方法查询
for doc, meta, distance, doc_id in zip(results["documents"][0],results["metadatas"][0],results["distances"][0],results["ids"][0],):
    print(f"{doc[:50]}..., metadata={meta}, distance={distance}, id={doc_id}")

{'ids': [['example_2', 'example_5', 'example_1']], 'embeddings': None, 'documents': [['什么是RAG', '什么是RAG', 'RAG 是一种结合了检索和生成的自然语言处理技术。']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'topic': 'rag', 'source': 'example.txt'}, {'topic': 'rag', 'source': 'example.txt'}, {'source': 'example.txt', 'topic': 'rag'}]], 'distances': [[0.04121924936771393, 0.04121924936771393, 0.6814790964126587]]}
什么是RAG..., metadata={'topic': 'rag', 'source': 'example.txt'}, distance=0.04121924936771393, id=example_2
什么是RAG..., metadata={'topic': 'rag', 'source': 'example.txt'}, distance=0.04121924936771393, id=example_5
RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'source': 'example.txt', 'topic': 'rag'}, distance=0.6814790964126587, id=example_1


### 5.2 similarity_search：简单 Top-K

`similarity_search()` 返回的是 `list[Document]`。

`filter` 可以按照 metadata 做过滤。

In [ ]:
docs_found = db.similarity_search(
    query="RAG是什么",
    k=3,
    filter={"source": "example.txt"},
)
print(docs_found)  # 输出是一个List，每个元素是Document对象 对象需要用.去访问属性
for doc in docs_found:
    print(f"{doc.page_content[:50]}..., metadata={doc.metadata}, id={doc.id}")

[Document(id='example_2', metadata={'topic': 'rag', 'source': 'example.txt'}, page_content='什么是RAG'), Document(id='example_5', metadata={'topic': 'rag', 'source': 'example.txt'}, page_content='什么是RAG'), Document(id='example_1', metadata={'topic': 'rag', 'source': 'example.txt'}, page_content='RAG 是一种结合了检索和生成的自然语言处理技术。')]
什么是RAG..., metadata={'topic': 'rag', 'source': 'example.txt'}, id=example_2
什么是RAG..., metadata={'topic': 'rag', 'source': 'example.txt'}, id=example_5
RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'topic': 'rag', 'source': 'example.txt'}, id=example_1


### 5.3 similarity_search_with_score：Top-K + 分数

`similarity_search_with_score()` 返回的是 `(Document, score)` 组成的列表。

在 Chroma 中，这个 score 经常表示 distance。一般可以先按“越小越相近”理解，具体含义和 collection 的距离度量有关。

In [35]:
docs_with_score = db.similarity_search_with_score(
    query="RAG是什么",
    k=3,
    filter={"source": "example.txt"},
)
print(docs_with_score)  

first_result = docs_with_score[0]
print(first_result)

for doc_obj, score in docs_with_score:
    print(f"{doc_obj.page_content[:50]}..., metadata={doc_obj.metadata}, distance={score}, id={doc_obj.id}")

[(Document(id='example_2', metadata={'topic': 'rag', 'source': 'example.txt'}, page_content='什么是RAG'), 0.04121926799416542), (Document(id='example_5', metadata={'source': 'example.txt', 'topic': 'rag'}, page_content='什么是RAG'), 0.04121926799416542), (Document(id='example_1', metadata={'source': 'example.txt', 'topic': 'rag'}, page_content='RAG 是一种结合了检索和生成的自然语言处理技术。'), 0.6814790964126587)]
(Document(id='example_2', metadata={'topic': 'rag', 'source': 'example.txt'}, page_content='什么是RAG'), 0.04121926799416542)
什么是RAG..., metadata={'topic': 'rag', 'source': 'example.txt'}, distance=0.04121926799416542, id=example_2
什么是RAG..., metadata={'source': 'example.txt', 'topic': 'rag'}, distance=0.04121926799416542, id=example_5
RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'source': 'example.txt', 'topic': 'rag'}, distance=0.6814790964126587, id=example_1


### 5.4 MMR：相关性 + 多样性

`max_marginal_relevance_search()` 会先取一批候选，再在候选中平衡相关性和多样性。

- `k`：最终返回多少条。
- `fetch_k`：内部先召回多少候选。
- `lambda_mult`：相关性和多样性的权重，越接近 1 越偏相关性，越接近 0 越偏多样性。
- `filter`: metedata过滤

In [ ]:
docs_mmr = db.max_marginal_relevance_search(
    query="RAG是什么",
    k=3,
    fetch_k=10,
    lambda_mult=0.5,
    filter={"source": "example.txt"},
)

for doc in docs_mmr:
    print(f"{doc.page_content[:50]}..., metadata={doc.metadata}, id={doc.id}")

## 6. Retriever 封装

Retriever 是 LangChain 对检索能力的统一封装。后续进入完整 RAG Chain 时，通常不是直接把 `db` 传给链，而是把 `retriever` 作为检索组件。

`as_retriever()` 的底层仍然是向量检索，只是统一成了 `invoke(query)` 这种接口。‘

底层就是 db.similarity_search

In [ ]:
query = "RAG是什么"

retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"source": "example.txt"},
    },
)

retrieved_docs = retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.page_content, doc.metadata, doc.id)

In [ ]:
# MMR retriever 写法。
mmr_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5,
        "filter": {"source": "example.txt"},
    },
)

retrieved_docs = mmr_retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.page_content, doc.metadata, doc.id)

## 7. 当前阶段已经覆盖的能力

到这里，你已经完成了 LangChain + Chroma 入门阶段的核心闭环：

1. 获取 embedding 模型。
2. 使用 `embed_query()` 和 `embed_documents()`。
3. 理解普通字符串和 `Document` 的区别。
4. 创建或连接 Chroma collection。
5. 使用 `from_texts()` 和 `from_documents()` 创建并写入数据。
6. 使用 `add_documents()` 追加数据。
7. 使用稳定 ID 支持更新、删除和去重。
8. 查看 collection、查看数据、按 ID 查询数据。
9. 使用底层 Chroma query 理解返回结构。
10. 使用 `similarity_search()` 做 Top-K 检索。
11. 使用 `similarity_search_with_score()` 查看 distance/score。
12. 使用 MMR 做兼顾相关性和多样性的检索。
13. 使用 metadata filter 做过滤检索。
14. 使用 retriever 作为后续 RAG Chain 的标准检索接口。

## 8. 下一阶段建议

基础入库和检索先到这里是合理的。下一阶段建议只补少量关键内容：

- chunking：把长文档切成适合检索的片段。
- 批量 JSONL 入库：把真实数据集批量转成 `Document`。
- 稳定 ID / 去重策略：避免重复导入。
- 检索效果评估：观察 query、召回文档、distance、metadata 是否符合预期。
- 接入 LLM：把 retriever 返回的文档拼成 context，进入完整 RAG 回答流程。